### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="aps_failure",
    dataset_year="2016",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5V60Q",
    download_description="""
mkdir -p local-data-warehouse/aps_failure \
&& wget -P local-data-warehouse/aps_failure/ https://archive.ics.uci.edu/static/public/414/ida2016challenge.zip \
&& unzip local-data-warehouse/aps_failure/ida2016challenge.zip -d local-data-warehouse/aps_failure/ \
&& rm local-data-warehouse/aps_failure/ida2016challenge.zip
""",
    # References
    academic_reference_bibtex="""@misc{ida2016challenge,
  author       = {{IDA2016Challenge}},
  title        = {IDA2016Challenge [Dataset]},
  year         = {2016},
  howpublished = {\\url{https://doi.org/10.24432/C5V60Q}},
  note         = {UCI Machine Learning Repository},
}
""",
    academic_reference_bibtex_key="ida2016challenge",
    license="CC BY 4.0",
    data_tags=["ForcedIIDFromTemporal", "Anonymized"],
    curation_comments="""
  - We combined the original training and testing data into a single dataset.
  - We renamed the target feature to "AirPressureSystemFailure".
  - We converted "na" strings to real NaN/missing values, making the data numeric.
  - Anomaly: we cannot determine the data types of the features.
  - Anomaly: some features are bins of histograms (see original data description).
  - Anomaly: the original task used a cost matrix for evaluation.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="AirPressureSystemFailure",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="AirPressureSystemFailure",
)

## Preprocessing

In [ ]:
import pandas as pd

# remove the leading text before the header form both the training and test data.
df = pd.concat(
    [
        pd.read_csv(f"{dataset_mold.path}/to_uci/aps_failure_training_set.csv", skiprows=20, na_values="na"),
        pd.read_csv(f"{dataset_mold.path}/to_uci/aps_failure_test_set.csv", skiprows=20, na_values="na"),
    ]
)

df.rename(columns={"class": "AirPressureSystemFailure"}, inplace=True)
df["AirPressureSystemFailure"] = df["AirPressureSystemFailure"].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()